# 04 — GARCH(1,1)–Merton

**What this notebook is:** a complete start-to-finish guide for discrete GARCH volatility **plus** Merton jumps, with an interactive Monte Carlo playground.

**Related:** [`02_merton.ipynb`](02_merton.ipynb) (jumps)

**Your project data:** `../data/equity/prices_clean.csv`, `log_returns_*.csv`, `summary_stats.csv`


## 1. Model idea

GARCH makes **today’s variance a function of yesterday’s shock and yesterday’s variance**. It is naturally a **daily discrete** model.

**Return this step:**

$$r_t = \mu\,\Delta t + \sigma_t Z_t + J_t$$

**Variance update (GARCH(1,1)):**

$$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$$

**Shock:** $\varepsilon_t = \sigma_t Z_t$, \quad $Z_t\sim N(0,1)$

**Price update:**

$$S_{t+\Delta t} = S_t \exp(r_t)$$

**Jumps $J_t$:** same compound-Poisson idea as Merton (often 0; occasionally a normal log-jump sum).

| Symbol | Meaning |
|--------|---------|
| $\omega$ | baseline variance floor |
| $\alpha$ | ARCH weight — reaction to yesterday’s shock² |
| $\beta$ | GARCH weight — persistence of vol |
| $\sigma_0$ | starting volatility (per step) |
| $\mu$ | drift (per year; scaled by $\Delta t$ in the return) |
| $\lambda,\mu_J,\sigma_J$ | jump intensity and size |

**Stability:** $\alpha+\beta < 1$. Long-run variance $\bar\sigma^2 = \omega/(1-\alpha-\beta)$.


## 2. End-to-end workflow

| Step | What you do | Output |
|------|-------------|--------|
| **1. Collect prices** | Adjusted closes for ticker / regime | Price series |
| **2. Log returns** | $r_t=\ln(S_t/S_{t-1})$ | Daily return series |
| **3. Fit GARCH(1,1)** | Estimate $\omega,\alpha,\beta$ (and $\mu$) from returns | Vol parameters |
| **4. Set $\sigma_0$** | Use fitted conditional vol at sample end, or $\sqrt{\bar\sigma^2}$ | Starting vol |
| **5. Estimate jumps** | Threshold / Merton step on residuals or raw returns | $\lambda,\mu_J,\sigma_J$ |
| **6. Set design** | $S_0$, $T$, steps (usually daily), paths | Grid |
| **7. Simulate** | Each day: update $\sigma_t^2$, draw $Z_t$ and jumps, update $S$ | Price + conditional vol paths |
| **8. Use paths** | Clustering diagnostics, pricing, regime comparison | Research outputs |

GARCH is the most “data-native” of the four for daily equity returns because the recursion matches daily steps.


## 3. How to calculate parameters from historical data

### A. Log returns (daily)

$$r_t = \ln(S_t/S_{t-1})$$

Use `../data/equity/log_returns_by_regime.csv` filtered to ticker + regime.

### B. Fit GARCH(1,1) — estimate main parameters (once)

Standard approach: **maximum likelihood** for

$$r_t = \mu_{\text{day}} + \varepsilon_t,\quad \varepsilon_t=\sigma_t Z_t,\quad \sigma_t^2=\omega+\alpha\varepsilon_{t-1}^2+\beta\sigma_{t-1}^2$$

In Python (later calibration step), packages like `arch` do this:

```python
# sketch — run in a calibration script, not required for the playground
from arch import arch_model
am = arch_model(returns * 100, vol="Garch", p=1, q=1, dist="normal")
res = am.fit(disp="off")
# res.params → mu, omega, alpha[1], beta[1]
```

Units note: many optimizers prefer returns in **percent**. Convert carefully back to decimal variance for simulation.

### C. Read off $\omega,\alpha,\beta$ (once)

From the fit:

- $\omega$ — intercept (small for daily decimal returns, e.g. order $10^{-6}$)
- $\alpha$ — shock sensitivity (clustering)
- $\beta$ — persistence (often $0.85$–$0.95$)

Check $\hat\alpha+\hat\beta < 1$.

### D. Long-run variance and $\sigma_0$

$$\bar\sigma^2 = \frac{\omega}{1-\alpha-\beta}$$

- $\sigma_0$: last conditional $\sigma$ from the fit, or $\sqrt{\bar\sigma^2}$.

### E. Drift $\mu$ (once)

Annualize the fitted daily mean: $\hat\mu = \hat\mu_{\text{day}}\times 252$, matching how this notebook scales `mu * dt`.

### F. Jump parameters (once)

After a GARCH fit, large **standardized** residuals $|\varepsilon_t/\sigma_t|$ can flag jumps; then estimate $\lambda,\mu_J,\sigma_J$ as in Merton. Simpler: threshold raw returns, then optionally refit GARCH on non-jump days.

### G. Sanity checks from data

| Check | What you want |
|-------|----------------|
| $\alpha+\beta$ | $<1$, often close to 1 (persistent vol) |
| Crisis vs normal | higher $\omega$ or higher effective vol level in crisis |
| $\alpha$ | larger ⇒ stronger reaction to big moves |


## 4. Constant vs path-updating parameters

### Calibrated once (fixed for the whole run)

| Parameter | Role | Updates during a path? |
|-----------|------|------------------------|
| $\mu$ | drift | **No** |
| $\omega,\alpha,\beta$ | GARCH law | **No** — the *rule* is fixed |
| $\sigma_0$ | initial vol | **No** (starting value only) |
| $\lambda,\mu_J,\sigma_J$ | jumps | **No** |
| $S_0,T,\Delta t$ | design | **No** |

### Evolve along each Monte Carlo path

| Quantity | Role | Updates during a path? |
|----------|------|------------------------|
| $\sigma_t^2$ (or $\sigma_t$) | conditional variance | **Yes** — via GARCH recursion |
| $\varepsilon_t=\sigma_t Z_t$ | shock | **Yes** — feeds **next** day’s variance |
| $S_t$ | price | **Yes** |
| $Z_t$, jumps | randomness | **Yes** — redrawn each step |

Variance is **deterministic given past shocks** (no separate vol Brownian motion). Randomness enters through $Z_t$ (and jumps); that shock then changes tomorrow’s $\sigma$.

### One simulation step

1. Keep $(\mu,\omega,\alpha,\beta,\sigma_0,\lambda,\mu_J,\sigma_J)$ fixed.
2. Set $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2$.
3. Draw $Z_t$; set $\varepsilon_t=\sigma_t Z_t$; add jump contribution $J_t$.
4. $r_t = \mu\Delta t + \varepsilon_t + J_t$, then $S\leftarrow S\exp(r_t)$.
5. Store $\varepsilon_t$ for the next step’s variance update.


## 5. Interactive playground

Raise $\alpha$ to strengthen clustering; raise $\beta$ for longer vol memory (keep $\alpha+\beta<1$). Add jumps with $\lambda$ / $\sigma_J$. The middle panel shows conditional $\sigma_t$ along paths.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_garch_merton(
    mu, omega, alpha, beta, sigma0,
    lam, mu_j, sigma_j,
    S0, T, n_steps, n_paths, seed=42,
):
    rng = np.random.default_rng(seed)
    dt = T / n_steps  # treat each step as dt years; scale drift/intensity
    # GARCH params assumed on the step frequency used in the slider

    S = np.full(n_paths, S0, dtype=float)
    var = np.full(n_paths, sigma0**2, dtype=float)
    paths = np.empty((n_paths, n_steps + 1))
    vol_paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = S
    vol_paths[:, 0] = np.sqrt(var)
    log_rets = np.empty((n_paths, n_steps))
    eps_prev = np.zeros(n_paths)

    for i in range(n_steps):
        var = omega + alpha * eps_prev**2 + beta * var
        var = np.maximum(var, 1e-12)
        sigma = np.sqrt(var)

        z = rng.standard_normal(n_paths)
        eps = sigma * z

        n_jumps = rng.poisson(lam * dt, size=n_paths)
        jump = np.zeros(n_paths)
        mask = n_jumps > 0
        jump[mask] = (
            n_jumps[mask] * mu_j
            + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(mask.sum())
        )

        r = mu * dt + eps + jump
        S = S * np.exp(r)
        paths[:, i + 1] = S
        vol_paths[:, i + 1] = sigma
        log_rets[:, i] = r
        eps_prev = eps

    t = np.linspace(0, T, n_steps + 1)
    return t, paths, vol_paths, log_rets

def plot_garch_merton(
    mu=0.08, omega=1e-6, alpha=0.08, beta=0.90, sigma0=0.012,
    lam=0.4, mu_j=-0.04, sigma_j=0.08,
    S0=100.0, T=1.0, n_steps=500, n_paths=1000,
):
    # keep stationarity-ish: alpha + beta < 1
    if alpha + beta >= 0.999:
        beta = max(0.0, 0.998 - alpha)

    t, paths, vol_paths, log_rets = simulate_garch_merton(
        mu, omega, alpha, beta, sigma0, lam, mu_j, sigma_j, S0, T, n_steps, n_paths
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.8)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean")
    axes[0].set_title("Price paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].plot(t, vol_paths.T, alpha=0.35, lw=0.8)
    axes[1].plot(t, vol_paths.mean(axis=0), color="black", lw=2)
    axes[1].set_title("GARCH conditional σ")
    axes[1].set_xlabel("years")

    axes[2].hist(log_rets.ravel(), bins=80, density=True, alpha=0.75, color="slateblue")
    axes[2].set_title("Step log returns")
    axes[2].set_xlabel("log return")

    fig.suptitle(
        f"ω={omega:.1e}, α={alpha:.2f}, β={beta:.2f}, λ={lam:.2f}  (α+β={alpha+beta:.3f})",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_garch_merton,
    mu=FloatSlider(value=0.08, min=-0.10, max=0.30, step=0.01, description="μ/year"),
    omega=FloatSlider(value=1e-6, min=1e-8, max=5e-5, step=1e-7, description="ω", readout_format=".1e"),
    alpha=FloatSlider(value=0.08, min=0.0, max=0.40, step=0.01, description="α"),
    beta=FloatSlider(value=0.90, min=0.50, max=0.98, step=0.01, description="β"),
    sigma0=FloatSlider(value=0.012, min=0.002, max=0.05, step=0.001, description="σ0"),
    lam=FloatSlider(value=0.40, min=0.0, max=3.0, step=0.1, description="λ"),
    mu_j=FloatSlider(value=-0.04, min=-0.30, max=0.15, step=0.01, description="μ_J"),
    sigma_j=FloatSlider(value=0.08, min=0.01, max=0.40, step=0.01, description="σ_J"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="T"),
    n_steps=IntSlider(value=500, min=50, max=2000, step=10, description="steps"),
    n_paths=IntSlider(value=1000, min=5, max=2000, step=5, description="paths"),
);